# WebShield: Automated Website Security Auditing and Risk Assessment Tool

**Purpose:** Check a website that you own or have explicit permission to test for selected HTTP security headers and basic CORS configuration issues. This is a lightweight educational checker, not a full penetration-testing tool.

Use only on systems you own or are authorized to assess. The scanner sends ordinary HTTP GET/OPTIONS requests and does not attempt exploitation, authentication bypass, or denial of service.

## 1. Install dependencies
Run the next cell once in your notebook environment. You can also install the dependencies with `pip install -r requirements.txt`.

In [ ]:
%pip install requests pandas

## 2. Import libraries and configure checks

In [ ]:
from urllib.parse import urlparse
from datetime import datetime, timezone
import requests
import pandas as pd
import re

SECURITY_HEADERS = {
    'Strict-Transport-Security': ('High', 'Use HTTPS and configure an appropriate HSTS policy.'),
    'Content-Security-Policy': ('High', 'Define a Content Security Policy appropriate for the application.'),
    'X-Content-Type-Options': ('Medium', 'Set X-Content-Type-Options: nosniff.'),
    'X-Frame-Options': ('Medium', 'Set X-Frame-Options to DENY or SAMEORIGIN, or use frame-ancestors in CSP.'),
    'Referrer-Policy': ('Low', 'Set a Referrer-Policy such as strict-origin-when-cross-origin.'),
    'Permissions-Policy': ('Low', 'Restrict browser features that the application does not need.')
}

REQUEST_TIMEOUT_SECONDS = 10
USER_AGENT = 'WebShield-Educational-Auditor/1.0'
print('WebShield is ready.')

## 3. Scanner functions
The scanner validates the URL, checks the response headers, and performs a simple CORS preflight-style OPTIONS request. CORS results are indicators for manual review, not proof of a vulnerability.

In [2]:
def normalize_url(url: str) -> str:
    """Validate and normalize an HTTP(S) URL."""
    url = url.strip()
    parsed = urlparse(url)
    if parsed.scheme not in ('http', 'https') or not parsed.hostname:
        raise ValueError('Enter a complete URL beginning with http:// or https://')
    if parsed.username or parsed.password:
        raise ValueError('Do not include usernames or passwords in the URL.')
    return url

def add_finding(rows, category, item, status, severity, evidence, recommendation):
    rows.append({
        'Category': category,
        'Check': item,
        'Status': status,
        'Severity': severity,
        'Evidence': str(evidence)[:500],
        'Recommendation': recommendation
    })
    REQUEST_TIMEOUT_SECONDS = 10
USER_AGENT = "WebShield-Educational-Auditor/1.0"

def scan_website(url: str, timeout: int = REQUEST_TIMEOUT_SECONDS):
    """Audit selected security headers and basic CORS behavior for an authorized target."""
    url = normalize_url(url)
    rows = []
    session = requests.Session()
    session.headers.update({'User-Agent': USER_AGENT})
    try:
        response = session.get(url, timeout=timeout, allow_redirects=True)
        final_url = response.url
        headers = response.headers
        is_https = urlparse(final_url).scheme == 'https'

        for header, (severity, recommendation) in SECURITY_HEADERS.items():
            present = header in headers and bool(headers.get(header, '').strip())
            if header == 'Strict-Transport-Security' and not is_https:
                add_finding(rows, 'Security Header', header, 'Review', 'High',
                            'Final response is not HTTPS; HSTS is meaningful only over HTTPS.',
                            'Enable HTTPS first, then configure HSTS where appropriate.')
            elif present:
                add_finding(rows, 'Security Header', header, 'Present', 'Info',
                            headers.get(header), 'Review the policy value to ensure it matches your application needs.')
            else:
                add_finding(rows, 'Security Header', header, 'Missing', severity,
                            'Header was not present in the final GET response.', recommendation)

        # Basic CORS behavior probe using a reserved example origin.
        test_origin = 'https://webshield-test.example'
        try:
            cors_response = session.options(
                final_url,
                headers={
                    'Origin': test_origin,
                    'Access-Control-Request-Method': 'GET',
                    'Access-Control-Request-Headers': 'content-type'
                },
                timeout=timeout,
                allow_redirects=False
            )
            acao = cors_response.headers.get('Access-Control-Allow-Origin', '')
            acac = cors_response.headers.get('Access-Control-Allow-Credentials', '').lower()
            if acao == '*' and acac == 'true':
                add_finding(rows, 'CORS', 'Wildcard origin with credentials', 'Review', 'High',
                            f'Access-Control-Allow-Origin={acao}; Access-Control-Allow-Credentials={acac}',
                            'Do not combine wildcard origins with credentialed requests; allow only trusted origins.')
            elif acao == test_origin and acac == 'true':
                add_finding(rows, 'CORS', 'Untrusted test origin accepted with credentials', 'Review', 'High',
                            f'Test origin reflected; credentials={acac}',
                            'Verify the origin allowlist and ensure arbitrary origins cannot receive credentialed access.')
            elif acao == '*':
                add_finding(rows, 'CORS', 'Wildcard allowed origin', 'Review', 'Low',
                            'Access-Control-Allow-Origin=*',
                            'Confirm that public cross-origin access is intentional and no sensitive data is exposed.')
            elif acao:
                add_finding(rows, 'CORS', 'Origin response', 'Observed', 'Info',
                            f'Access-Control-Allow-Origin={acao}; credentials={acac or "not set"}',
                            'Confirm allowed origins are intentional and narrowly scoped.')
            else:
                add_finding(rows, 'CORS', 'Test origin response', 'No allow-origin header', 'Info',
                            f'OPTIONS status={cors_response.status_code}',
                            'No CORS allow-origin header was observed in this probe; test actual application endpoints as needed.')
        except requests.RequestException as exc:
            add_finding(rows, 'CORS', 'OPTIONS probe', 'Could not test', 'Info', str(exc),
                        'Review CORS manually or test a known API endpoint that supports OPTIONS.')

        metadata = {
            'Target URL': url,
            'Final URL': final_url,
            'HTTP Status': response.status_code,
            'Checked At (UTC)': datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC'),
            'Redirects Followed': len(response.history)
        }
        return metadata, pd.DataFrame(rows)
    except requests.RequestException as exc:
        raise RuntimeError(f'Could not reach target: {exc}') from exc
       

NameError: name 'REQUEST_TIMEOUT_SECONDS' is not defined

## 4. Run a scan
Replace the sample URL with a website you own or are explicitly authorized to test. For a classroom demo, use a local test application or a controlled lab target.

In [1]:
target_url = 'https://example.com'  # Replace with your authorized test target
metadata, results = scan_website(target_url)
print('Scan summary')
for key, value in metadata.items():
    print(f'{key}: {value}')
display(results)

NameError: name 'scan_website' is not defined

## 5. Summary and export report
The CSV file contains findings that can be attached to the project report. Severity is a triage aid, not a formal CVSS score.

In [ ]:
severity_counts = results['Severity'].value_counts().rename_axis('Severity').reset_index(name='Count')
display(severity_counts)
safe_name = re.sub(r'[^A-Za-z0-9_-]+', '_', urlparse(metadata['Final URL']).hostname or 'target')
results.to_csv(f'webshield_findings_{safe_name}.csv', index=False)
pd.DataFrame([metadata]).to_csv(f'webshield_scan_metadata_{safe_name}.csv', index=False)
print('CSV reports exported to the current notebook folder.')

## Limitations and responsible use
- Checks a selected set of response headers and one basic CORS probe only.
- Does not crawl a site, test authentication, discover all endpoints, or prove exploitability.
- Header presence does not guarantee a correct policy; inspect values and application behavior.
- CORS behavior can differ by route, method, origin, and credentials. Manually validate findings.
- Run only against systems you own or have permission to assess.